# Broadband Model Training with Weights & Biases Tracking

This notebook demonstrates how to train a broadband sky brightness model with
**real-time experiment tracking** using [Weights & Biases](https://wandb.ai/) (wandb).

**What's new compared to `01_broadband_training.ipynb`:**
- `WandbConfig` enables automatic scalar metric logging (`train/loss`, `val/loss`)
- `on_epoch_end` callback logs custom visualizations:
  - Per-band 2×2 diagnostic panels (scatter + residual) for V, g, r, z
- Metrics organized into wandb sections: `train/`, `val/`, `viz/`
- Checkpoint naming uses wandb run name for unique identification
- Optional: hyperparameter sweep example

**Requirements:**
```bash
pip install desisky[wandb]
# or: pip install desisky[all]
```

> **Note:** GPU training is recommended for faster convergence. For CUDA support: `pip install desisky[cuda12,wandb]`

## 1. Imports

In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import equinox as eqx
import matplotlib.pyplot as plt
import torch
from torch.utils.data import random_split
import wandb

from desisky.data import SkySpecVAC
from desisky.training import (
    SkyBrightnessDataset,
    NumpyLoader,
    BroadbandTrainer,
    TrainingConfig,
    WandbConfig,
    gather_full_data,
)
from desisky.training.wandb_utils import log_figure
from desisky.visualization import plot_broadband_band_panel

print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

/home/mdowicz/miniconda3/envs/desisky_dev/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


JAX version: 0.7.1
JAX devices: [CudaDevice(id=0), CudaDevice(id=1)]


## 2. wandb Login

In [2]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/mdowicz/.netrc.
wandb: Currently logged in as: mdowicz to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 3. Load Data

In [3]:
vac = SkySpecVAC(version="v1.0", download=True, verify=True)
wave, flux, meta = vac.load_moon_contaminated()

print(f"Loaded {len(meta)} moon-contaminated observations")
print(f"Metadata columns: {list(meta.columns)}")

Loaded 1931 moon-contaminated observations
Metadata columns: ['NIGHT', 'EXPID', 'TILEID', 'TILERA', 'TILEDEC', 'MJD', 'EXPTIME', 'AIRMASS', 'EBV', 'SEEING_ETC', 'TRANSPARENCY_GFA', 'SEEING_GFA', 'SKY_MAG_G_SPEC', 'SKY_MAG_R_SPEC', 'SKY_MAG_Z_SPEC', 'MOONFRAC', 'MOONALT', 'MOONSEP', 'SUNALT', 'SUNSEP', 'OBSALT', 'OBSAZ', 'SKY_MAG_V_SPEC', 'ECLIPSE_FRAC', 'SOLFLUX', 'GALLON', 'GALLAT', 'ECLLON', 'ECLLAT']


## 4. Create Dataset and DataLoaders

In [4]:
input_features = [
    'MOONSEP',
    'OBSALT',
    'MOONALT',
    'MOONFRAC',
    'TRANSPARENCY_GFA',
    'ECLIPSE_FRAC',
]

dataset = SkyBrightnessDataset(
    metadata=meta,
    flux=flux,
    input_features=input_features,
)

# 70/30 train/test split
dataset_size = len(dataset)
train_size = int(0.7 * dataset_size)
test_size = dataset_size - train_size

gen = torch.Generator().manual_seed(42)
train_set, test_set = random_split(dataset, [train_size, test_size], generator=gen)

batch_size = 64
train_loader = NumpyLoader(dataset=train_set, batch_size=batch_size, shuffle=True)
test_loader = NumpyLoader(dataset=test_set, batch_size=batch_size, shuffle=False)

print(f"Training: {len(train_set)} samples ({len(train_loader)} batches)")
print(f"Test: {len(test_set)} samples ({len(test_loader)} batches)")

Training: 1351 samples (22 batches)
Test: 580 samples (10 batches)


## 5. Create Model

In [5]:
model = eqx.nn.MLP(
    in_size=len(input_features),
    out_size=4,  # V, g, r, z
    width_size=128,
    depth=5,
    key=jax.random.PRNGKey(42),
)

print(f"Model: MLP(in={len(input_features)}, out=4, width=128, depth=5)")

Model: MLP(in=6, out=4, width=128, depth=5)


## 6. Configure Training + wandb

In [6]:
config = TrainingConfig(
    epochs=101,
    learning_rate=1e-4,
    loss="huber",
    huber_delta=0.25,
    save_best=True,
    run_name="broadband_moon_wandb",
    print_every=50,
    validate_every=1,
)

wconfig = WandbConfig(
    project="desisky-broadband",
    run_name=None,          # Auto-generate unique name
    tags=["broadband", "moon", "huber"],
    log_every=1,
    viz_every=50,
)

## 7. Define Visualization Callback

The callback logs a 2×2 diagnostic panel for each broadband band (V, g, r, z) every `viz_every` epochs:
- **Top-left:** Train scatter (observed vs predicted) with 1:1 line and ±σ rails
- **Bottom-left:** Test scatter (same format)
- **Top-right:** Train residual histogram
- **Bottom-right:** Test residual histogram

In [7]:
BAND_NAMES = ["V", "g", "r", "z"]

# Pre-extract test data for visualizations
X_train, y_train, _, _ = gather_full_data(train_loader)
X_test, y_test, _, _ = gather_full_data(test_loader)


def on_epoch_end(model, history, epoch):
    if epoch % wconfig.viz_every != 0:
        return

    for band_idx, band_name in enumerate(BAND_NAMES):
        fig = plot_broadband_band_panel(
            model,
            X_train, y_train,
            X_test, y_test,
            band_idx=band_idx,
            band_name=band_name,
        )
        log_figure(f"viz/{band_name}_panel", fig, epoch)
        plt.close(fig)


print(f"Callback defined. Logs 4 band panels every {wconfig.viz_every} epochs.")

Callback defined. Logs 4 band panels every 50 epochs.


## 8. Train with wandb Tracking

In [8]:
trainer = BroadbandTrainer(
    model, config,
    wandb_config=wconfig,
    on_epoch_end=on_epoch_end,
)

print("Starting training...")
model, history = trainer.train(train_loader, test_loader)

print(f"\nBest test loss: {history.best_test_loss:.6f} (epoch {history.best_epoch})")

Starting training...


  wandb run: https://wandb.ai/mdowicz/desisky-broadband/runs/6r7elj7i
Epoch    0 | Train: 4.621329 | Test: 4.428833 | Best: 4.428833
Epoch   50 | Train: 0.265467 | Test: 0.272081 | Best: 0.270247
Epoch  100 | Train: 0.258645 | Test: 0.273123 | Best: 0.264232
  wandb run: https://wandb.ai/mdowicz/desisky-broadband/runs/6r7elj7i


epoch,▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇████
train/loss,█▅▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/loss,█▇▇▅▅▄▄▃▃▃▃▃▃▂▃▂▂▄▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▄▁▃▁▁▂
epoch,100
train/loss,0.25864
val/loss,0.27312



Best test loss: 0.264232 (epoch 99)


## 9. Summary

All metrics and plots are available on your wandb dashboard, organized into three sections:

**`train/` — Training scalars (every epoch):**
- `train/loss` — Huber loss (or L2, depending on config)

**`val/` — Validation scalars (every epoch):**
- `val/loss`

**`viz/` — Figures (every `viz_every` epochs):**
- `viz/V_panel`, `viz/g_panel`, `viz/r_panel`, `viz/z_panel` — per-band 2×2 scatter + residual diagnostic panels

## 10. Optional: Hyperparameter Sweep

In [11]:
sweep_config = {
    "method": "grid",
    "metric": {"name": "val/loss", "goal": "minimize"},
    "parameters": {
        "learning_rate": {"value": 1e-4}, #[5e-5, 1e-4, 3e-4]},
        "huber_delta": {"values": [0.1, 0.25, 0.5]},
    },
}


def train_sweep(config=None):
    # Must init before accessing wandb.config (sweep agent populates it)
    wandb.init()

    sweep_train_config = TrainingConfig(
        epochs=101,
        learning_rate=wandb.config.learning_rate,
        loss="huber",
        huber_delta=wandb.config.huber_delta,
        save_best=False,
        print_every=100,
        validate_every=1,
    )

    sweep_wconfig = WandbConfig(
        project="desisky-broadband",
        tags=["sweep", "lr-huber"],
        log_every=1,
        viz_every=50,
    )

    sweep_model = eqx.nn.MLP(
        in_size=len(input_features),
        out_size=4,
        width_size=128,
        depth=5,
        key=jax.random.PRNGKey(42),
    )
    sweep_trainer = BroadbandTrainer(
        sweep_model, sweep_train_config,
        wandb_config=sweep_wconfig,
        on_epoch_end=on_epoch_end,
    )
    sweep_trainer.train(train_loader, test_loader)

# Uncomment to commence wandb sweep
sweep_id = wandb.sweep(sweep_config, project="desisky-broadband")
wandb.agent(sweep_id, train_sweep, count=2)

Create sweep with ID: zouznie2
Sweep URL: https://wandb.ai/mdowicz/desisky-broadband/sweeps/zouznie2


wandb: Agent Starting Run: hyty90ht with config:
wandb: 	huber_delta: 0.1
wandb: 	learning_rate: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/mdowicz/.netrc.


  wandb run: https://wandb.ai/mdowicz/desisky-broadband/runs/hyty90ht
Epoch    0 | Train: 1.857074 | Test: 1.778809 | Best: 1.778809
Epoch  100 | Train: 0.113052 | Test: 0.112717 | Best: 0.112717
  wandb run: https://wandb.ai/mdowicz/desisky-broadband/runs/hyty90ht


epoch,▁▁▁▁▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇█████
train/loss,█▅▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,100
train/loss,0.11305
val/loss,0.11272


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: hi1owhfa with config:
wandb: 	huber_delta: 0.25
wandb: 	learning_rate: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/mdowicz/.netrc.


  wandb run: https://wandb.ai/mdowicz/desisky-broadband/runs/hi1owhfa
Epoch    0 | Train: 4.619899 | Test: 4.427898 | Best: 4.427898
Epoch  100 | Train: 0.262280 | Test: 0.263667 | Best: 0.263667
  wandb run: https://wandb.ai/mdowicz/desisky-broadband/runs/hi1owhfa


epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇███
train/loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/loss,█▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▃▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁
epoch,100
train/loss,0.26228
val/loss,0.26367
